# CeyMo flip-fix quick check (Kaggle)
Settings: **Accelerator = GPU P100** (or T4), **Internet = On**. Add Input: `experiments-code` and your CeyMo dataset (raw CeyMo with train/ and test/, or converted ceymo_official).
Run with **Save Version → Save & Run All (Commit)** so it runs up to 12 h in the background and keeps the output.

In [ ]:
# ---- settings: edit here ----
IMGSZ = 640        # 640 first; 1280 needs ~2 sessions on P100/T4
BATCH = 8          # use 4 for IMGSZ=1280
VARIANTS = 'baseline'
SEEDS = '0'


In [ ]:
!nvidia-smi
!pip install -q ultralytics==8.4.90 pyyaml opencv-python matplotlib

In [ ]:
import glob, os, shutil, subprocess
W = '/kaggle/working/experiments'
D = '/kaggle/temp/ceymo_official'

# locate uploaded code anywhere under /kaggle/input
run_all = sorted(glob.glob('/kaggle/input/**/experiments/run_all.sh', recursive=True))
assert run_all, 'experiments code not found in /kaggle/input'
if not os.path.exists(W):
    shutil.copytree(os.path.dirname(run_all[0]), W)
os.chmod(f'{W}/run_all.sh', 0o755)

if not os.path.exists(f'{D}/data.yaml'):
    conv = sorted(glob.glob('/kaggle/input/**/ceymo_official/data.yaml', recursive=True))
    if conv:  # already converted dataset uploaded: copy it and fix the absolute path
        shutil.copytree(os.path.dirname(conv[0]), D)
        lines = open(f'{D}/data.yaml').read().splitlines()
        open(f'{D}/data.yaml', 'w').write('\n'.join(f'path: {D}' if l.startswith('path:') else l for l in lines) + '\n')
    else:     # raw CeyMo uploaded: find the folder holding train/ and test/ and convert (same seeded split)
        # train/ and test/ may be extracted side by side or into separate folders: find each one
        tr = sorted(glob.glob('/kaggle/input/**/train/polygon_annotations', recursive=True))
        te = sorted(glob.glob('/kaggle/input/**/test/polygon_annotations', recursive=True))
        assert tr and te, 'CeyMo not found: need .../train/polygon_annotations and .../test/polygon_annotations'
        R = '/kaggle/temp/ceymo_raw'; os.makedirs(R, exist_ok=True)
        for name, p in (('train', tr[0]), ('test', te[0])):
            if not os.path.exists(f'{R}/{name}'):
                os.symlink(os.path.dirname(p), f'{R}/{name}')
            print(name, '->', os.path.dirname(p), len(os.listdir(f'{R}/{name}/images')), 'images')
        raw = [R]
        subprocess.run(['python', f'{W}/convert_ceymo.py', '--ceymo', raw[0], '--out', D], check=True)
print(open(f'{D}/data.yaml').read())

# resume: if a previous notebook's output is attached as input, copy its runs/results back
for d in glob.glob('/kaggle/input/**/experiments/runs_flip*', recursive=True) + \
         glob.glob('/kaggle/input/**/experiments/results_flip*', recursive=True):
    dst = os.path.join(W, os.path.basename(d))
    if not os.path.exists(dst):
        shutil.copytree(d, dst); print('resumed from', d)


In [ ]:
%cd /kaggle/working/experiments
!VARIANTS="{VARIANTS}" SEEDS="{SEEDS}" TABLES=0 WORKERS=2 BATCH={BATCH} IMGSZ={IMGSZ} DEVICE=0 \
    DATA_DIR=/kaggle/temp/ceymo_official ./run_all.sh 2>&1 | tee quick_{IMGSZ}.log

In [ ]:
%cd /kaggle/working/experiments
!python per_class_report.py results_flipswap_{IMGSZ}
# old baseline for comparison: mIoU 67.5, LA 38.9, SRA 30.5